# LIGHTNING TEST

### We had flash-attention compatibility issues with the new runtime so mask it out.  
uninstalling it didn't work because it was baked into the env.

In [0]:
import os
# Prevent TensorFlow spam
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# Disable optional backends that trigger builds
os.environ["DISABLE_DEEPSPEED"] = "1"
os.environ["DISABLE_TRITON"] = "1"
os.environ["DISABLE_FLASH_ATTENTION"] = "1"
os.environ["XFORMERS_DISABLED"] = "1"

Have to run above before importing anything like transformers or lightning. So re

In [0]:
import torch, sys, subprocess, os
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda (torch):", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())

# nvcc (nice-to-have; may not exist on managed images)
try:
    print("nvcc:", subprocess.check_output(["nvcc", "--version"]).decode().strip().splitlines()[-1])
except Exception as e:
    print("nvcc: n/a", e)

In [0]:
import os, socket, torch, time
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.strategies import DDPStrategy

# --- Dataset ---
class ToyDataset(torch.utils.data.Dataset):
    def __init__(self, n=50_000, in_dim=784, n_classes=10):
        g = torch.Generator().manual_seed(0)
        self.x = torch.randn(n, in_dim, generator=g)
        self.y = torch.randint(0, n_classes, (n,), generator=g)
    def __len__(self): return self.x.size(0)
    def __getitem__(self, i): return self.x[i], self.y[i]

# --- LightningModule ---
class LitMLP(pl.LightningModule):
    def __init__(self, in_dim=784, hidden=512, n_classes=10, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes)
        )
        self.loss = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss(self(x), y)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)

# --- DataModule ---
class ToyDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=128):
        super().__init__()
        self.batch_size = batch_size

    def setup(self, stage=None):
        ds = ToyDataset()
        n_train = int(0.8 * len(ds))
        self.train_ds, self.val_ds = random_split(ds, [n_train, len(ds)-n_train])

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, num_workers=2, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, num_workers=2, pin_memory=True)

# --- Training entry point ---
def train_lightning():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    host = socket.gethostname()

    if rank == 0:
        print(f"🌍 Starting Lightning DDP: {world_size} total processes")

    # Setup trainer
    model = LitMLP()
    dm = ToyDataModule(batch_size=128)

    trainer = Trainer(
        accelerator="gpu",
        devices=1,                # 1 GPU per process
        strategy=DDPStrategy(find_unused_parameters=False),
        max_epochs=3,
        enable_progress_bar=(rank == 0),
        log_every_n_steps=20,
    )

    print(f"🧠 Rank {rank:02d} on {host} using GPU {local_rank}")
    trainer.fit(model, datamodule=dm)

    if rank == 0:
        print("✅ Lightning DDP training complete.")

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor

TorchDistributor(
    num_processes=16,  # 4 nodes × 4 GPUs
    local_mode=False,
    use_gpu=True
).run(train_lightning)